# 12 - Grover's Search

Quadratic speedup for unstructured search using amplitude amplification.

**Concepts:** Oracle, diffusion operator, amplitude amplification

In [ ]:
import quantsdk as qs
import math

## 2-Qubit Grover (Search 4 items)

In [ ]:
def grover_2qubit(target: str = "11") -> qs.Circuit:
    """Grover's search for a 2-bit target."""
    circuit = qs.Circuit(2, name=f"grover-{target}")
    
    # Superposition
    circuit.h(0).h(1)
    
    # Oracle: flip phase of target
    if target[0] == '0': circuit.x(0)
    if target[1] == '0': circuit.x(1)
    circuit.cz(0, 1)
    if target[0] == '0': circuit.x(0)
    if target[1] == '0': circuit.x(1)
    
    # Diffusion
    circuit.h(0).h(1)
    circuit.x(0).x(1)
    circuit.cz(0, 1)
    circuit.x(0).x(1)
    circuit.h(0).h(1)
    
    circuit.measure_all()
    return circuit

# Search for each possible target
for target in ['00', '01', '10', '11']:
    circuit = grover_2qubit(target)
    result = qs.run(circuit, shots=1000, seed=42)
    print(f"Target={target}: found={result.most_likely} (P={result.probabilities.get(target, 0):.3f})")

## 3-Qubit Grover (Search 8 items)

In [ ]:
def grover_3qubit(target: str = "101") -> qs.Circuit:
    """Grover's search for a 3-bit target."""
    n = 3
    num_iterations = int(math.pi / 4 * math.sqrt(2**n))
    circuit = qs.Circuit(n, name=f"grover-{target}")
    
    # Superposition
    for i in range(n):
        circuit.h(i)
    
    for _ in range(num_iterations):
        # Oracle
        for i in range(n):
            if target[i] == '0':
                circuit.x(i)
        circuit.ccz(0, 1, 2)
        for i in range(n):
            if target[i] == '0':
                circuit.x(i)
        
        # Diffusion
        for i in range(n): circuit.h(i)
        for i in range(n): circuit.x(i)
        circuit.ccz(0, 1, 2)
        for i in range(n): circuit.x(i)
        for i in range(n): circuit.h(i)
    
    circuit.measure_all()
    return circuit

target = "101"
circuit = grover_3qubit(target)
result = qs.run(circuit, shots=1000, seed=42)

print(f"Searching for: {target}")
print(f"Found: {result.most_likely}")
print(f"Probability: {result.probabilities.get(target, 0):.3f}")
print(f"Iterations: {int(math.pi/4 * math.sqrt(8))}")
print(f"\nFull distribution:")
for bs, count in sorted(result.counts.items(), key=lambda x: -x[1]):
    print(f"  |{bs}>: {count/1000:.3f}")